In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
sentiment = pd.read_csv(r"C:\Users\ASUS\Desktop\Data\fear_greed_index.csv")
trades = pd.read_csv(r"C:\Users\ASUS\Desktop\Data\historical_data.csv")

sentiment.head()
trades.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


In [9]:
print("Sentiment shape:", sentiment.shape)
print("Trades shape:", trades.shape)

Sentiment shape: (2644, 4)
Trades shape: (211224, 16)


In [11]:
print("\nMissing values:\n", sentiment.isnull().sum())
print("\nDuplicates:", sentiment.duplicated().sum())
print("\nMissing values:\n", trades.isnull().sum())
print("\nDuplicates:", trades.duplicated().sum())


Missing values:
 timestamp         0
value             0
classification    0
date              0
dtype: int64

Duplicates: 0

Missing values:
 Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64

Duplicates: 0


In [18]:
sentiment['date'] = pd.to_datetime(sentiment['date'], dayfirst=True)
trades['Timestamp'] = pd.to_datetime(trades['Timestamp'])


In [22]:
sentiment['date'] = pd.to_datetime(sentiment['date'], dayfirst=True, errors='coerce')
print(sentiment['date'].dtype)

datetime64[s]


In [33]:
sentiment['date'] = pd.to_datetime(sentiment['date'])
print(sentiment['date'].dtype)
sentiment['date'] = sentiment['date'].dt.date
sentiment['date'].head(10)

datetime64[s]


0    2018-02-01
1    2018-02-02
2    2018-02-03
3    2018-02-04
4    2018-02-05
5    2018-02-06
6    2018-02-07
7    2018-02-08
8    2018-02-09
9    2018-02-10
Name: date, dtype: object

In [34]:
print(trades.columns)

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp', 'date'],
      dtype='str')


In [35]:
trades['Timestamp'] = pd.to_datetime(trades['Timestamp'])
trades['date'] = trades['Timestamp'].dt.date

In [36]:
df = trades.merge(sentiment[['date', 'classification']], on='date', how='left')

In [37]:
print(df['classification'].isnull().sum())
print(df.shape)

211224
(211224, 18)


In [40]:
df['classification'].value_counts()

Series([], Name: count, dtype: int64)

In [41]:
print("Trades date range:", trades['date'].min(), "to", trades['date'].max())
print("Sentiment date range:", sentiment['date'].min(), "to", sentiment['date'].max())

Trades date range: 1970-01-01 to 1970-01-01
Sentiment date range: 2018-02-01 to 2025-05-02


In [42]:
common_dates = set(trades['date']).intersection(set(sentiment['date']))

len(common_dates)

0

In [43]:
trades.columns = trades.columns.str.lower().str.replace(' ', '_')
sentiment.columns = sentiment.columns.str.lower().str.replace(' ', '_')

In [54]:
trades['timestamp_ist'] = pd.to_datetime(trades['timestamp_ist'], dayfirst=True)
sentiment['date'] = pd.to_datetime(sentiment['date'])


In [55]:
trades['date'] = trades['timestamp_ist'].dt.date
sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date

In [56]:
# Sort both
trades = trades.sort_values('date')
sentiment = sentiment.sort_values('date')

# Merge
df = trades.merge(sentiment[['date', 'classification']], on='date', how='left')

# Fill missing sentiment
df['classification'] = df['classification'].ffill()

In [57]:
df['classification'].value_counts()

classification
Fear             61837
Greed            50309
Extreme Greed    39992
Neutral          37686
Extreme Fear     21400
Name: count, dtype: int64

In [69]:
daily_pnl = df.groupby(['account', 'date'])['closed_pnl'].sum().reset_index()

daily_pnl.head()

,account,date,closed_pnl
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,0.0
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-17,0.0
2,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-18,0.0
3,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-22,-21227.0
4,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-26,1603.1


In [71]:
df['win'] = df['closed_pnl'] > 0

win_rate = df.groupby('account')['win'].mean().reset_index()

win_rate.head()

,account,win
0,0x083384f897ee0f19899168e3b1bec365f52a9012,0.359612
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,0.442720
2,0x271b280974205ca63b716753467d5a371de622ab,0.301917
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,0.438585
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,0.519914


In [72]:
avg_size = df.groupby('account')['size_usd'].mean().reset_index()

avg_size.head()

,account,size_usd
0,0x083384f897ee0f19899168e3b1bec365f52a9012,16159.576734
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,1653.226327
2,0x271b280974205ca63b716753467d5a371de622ab,8893.000898
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,507.626933
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,3138.894782


In [73]:
df['size_usd'].describe()

count    2.112240e+05
mean     5.639451e+03
std      3.657514e+04
min      0.000000e+00
25%      1.937900e+02
50%      5.970450e+02
75%      2.058960e+03
max      3.921431e+06
Name: size_usd, dtype: float64

In [78]:
trades_per_day = df.groupby('date').size().reset_index(name='trade_count')

trades_per_day.head()

,date,trade_count
0,2023-05-01,3
1,2023-12-05,9
2,2023-12-14,11
3,2023-12-15,2
4,2023-12-16,3


In [82]:
long_short = df['side'].value_counts()

ratio = long_short['BUY'] / long_short['SELL']

print("Long/Short Ratio:", ratio)

Long/Short Ratio: 0.9462627156125608


In [83]:
fg = df[df['classification'].isin(['Fear', 'Greed'])]

# Average PnL
fg.groupby('classification')['closed_pnl'].mean()

# Win rate
fg.groupby('classification')['win'].mean()

# Worst loss (drawdown proxy)
fg.groupby('classification')['closed_pnl'].min()

classification
Fear     -35681.74723
Greed   -117990.10410
Name: closed_pnl, dtype: float64

In [84]:
df['classification'].value_counts()

classification
Fear             61837
Greed            50309
Extreme Greed    39992
Neutral          37686
Extreme Fear     21400
Name: count, dtype: int64

In [85]:
df.groupby('classification')['size_usd'].mean()

classification
Extreme Fear     5349.731843
Extreme Greed    3112.251565
Fear             7816.109931
Greed            5737.962662
Neutral          4782.732661
Name: size_usd, dtype: float64

In [88]:
pd.crosstab(df['classification'], df['side'])

side,BUY,SELL
classification,,
Extreme Fear,10935,10465
Extreme Greed,17940,22052
Fear,30270,31567
Greed,24582,25727
Neutral,18969,18717


In [90]:
m = df['size_usd'].median()
df['size_type'] = df['size_usd'].apply(lambda x: 'High' if x > m else 'Low')

df.groupby('size_type')['closed_pnl'].mean()

size_type
High    93.116556
Low      4.381445
Name: closed_pnl, dtype: float64

In [91]:
counts = df['account'].value_counts()
df['trader_type'] = df['account'].apply(lambda x: 'Frequent' if counts[x] > 50 else 'Normal')

df.groupby('trader_type')['closed_pnl'].mean()

trader_type
Frequent    48.749001
Name: closed_pnl, dtype: float64

In [92]:
std = df.groupby('account')['closed_pnl'].std()
df['consistency'] = df['account'].apply(lambda x: 'Consistent' if std[x] < std.median() else 'Inconsistent')

df.groupby('consistency')['closed_pnl'].mean()

consistency
Consistent       16.697081
Inconsistent    100.326484
Name: closed_pnl, dtype: float64